# Late payment by large UK companies: a statistical analysis

**Patrick Gilliland**

**Question.** Is the share of invoices paid late lower in 2023 to 2024 than in 2018 to 2019, for the same large UK companies? By the share of invoices paid late I mean the data column `% Invoices not paid within agreed terms`.

**Data.** [Check when large businesses pay their suppliers](https://www.gov.uk/check-when-businesses-pay-invoices), the GOV.UK payment practices reports. Each row is one large company's report on how it paid its suppliers over a reporting period, and a large company must publish one at least twice a year. I use the export of all published reports from https://check-payment-practices.service.gov.uk/export/csv/, taken on 6 Oct 2026 and saved beside this notebook as `all_published_reports_20261006.csv`. Contains public sector information licensed under the Open Government Licence v3.0.

**Steps.** I check the file, load it without its free-text columns, describe it and list its anomalies before any analysis uses it. Then I build the analysis table from the Regime-1 reports of the paired companies, draw three visual models, train and evaluate the fitted model, run the hypothesis test and summarise what I found.

**Submission.** `analysis.ipynb`, the report as `Statistical_Analysis_Report.pdf` and `module_summary.pdf` (the same report under the two names the brief uses), `requirements.txt` and the dataset CSV.

## Setup

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

# I show every column, whole cell text and two decimal places with thousands
# separators, so the wide tables below read without scientific notation.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
DATA = Path("all_published_reports_20261006.csv")
EXPORT_URL = "https://check-payment-practices.service.gov.uk/export/csv/"
EXPECTED_SHA256 = (
    "f9f2552222190b9c5f1ca145cdc1771ac88ba81c38fcfef49fe64c3b696db535"
)

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} is not beside this notebook. It is the export of all "
        f"published reports from {EXPORT_URL}, taken on 6 Oct 2026, and a "
        "copy is in the submission zip. Save it here under this name."
    )
digest = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(f"File: {DATA}, {DATA.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")
# The export grows every day, so a new download is a different file and every
# figure below would change. I stop rather than analyse the wrong file.
if digest != EXPECTED_SHA256:
    raise ValueError(
        "This is not the export taken on 6 Oct 2026, which every figure in "
        "this notebook comes from. Use the copy in the submission zip."
    )
print("It matches the export taken on 6 Oct 2026.")

File: all_published_reports_20261006.csv, 103,043,357 bytes
SHA-256: f9f2552222190b9c5f1ca145cdc1771ac88ba81c38fcfef49fe64c3b696db535
It matches the export taken on 6 Oct 2026.


## Data handling

### Task 2: Load the dataset

In [3]:
# I take the date columns off the header, as the names that end in "date", so
# no column name is typed by hand. low_memory=False reads the file in one pass
# and gives each column one type; the whole file fits easily in memory here.
date_columns = [
    name
    for name in pd.read_csv(DATA, nrows=0).columns
    if name.endswith("date")
]
loaded = pd.read_csv(DATA, parse_dates=date_columns, low_memory=False)
print(f"As loaded: {len(loaded):,} rows, {loaded.shape[1]} columns")

# A text column whose median answer is longer than eight words holds written
# answers, not a code, a name or a yes/no. The analysis needs none of them, so
# I drop them here, before anything is shown, and never print one.
free_text = [
    name
    for name in loaded.select_dtypes(include="str").columns
    if loaded[name].str.split().str.len().median() > 8
]
df = loaded.drop(columns=free_text)
del loaded
print(f"Free-text columns dropped ({len(free_text)}):")
for name in free_text:
    print(f"  {name}")
print(f"Kept: {len(df):,} rows, {df.shape[1]} columns")

As loaded: 115,358 rows, 52 columns


Free-text columns dropped (10):
  Standard payment terms
  Payment terms have changed
  Suppliers notified of changes
  Maximum contractual payment period information
  Other information payment terms
  Description of specific circumstances for retention clauses
  Description of parity policy
  Retention clause money release description
  Description of stages for money release
  Dispute resolution process
Kept: 115,358 rows, 42 columns


In [4]:
# The flag columns hold only True, False or blank. Those with blanks load as
# object, so I give them pandas' nullable boolean type, which keeps all three.
flag_columns = [
    name
    for name in df.columns
    if df[name].dtype == object
    and set(df[name].dropna().unique()) <= {True, False}
]
df[flag_columns] = df[flag_columns].astype("boolean")
print(f"Flag columns converted from object to boolean: {len(flag_columns)}")

Flag columns converted from object to boolean: 11


In [5]:
df.head()

,Report Id,Policy Regime,Financial period start date,Start date,End date,Filing date,Company,Company number,Qualifying contracts in reporting period,Payments made in reporting period,Qualifying construction contracts in reporting period,Construction contracts have retention clauses,Average time to pay,Total value invoices paid within 30 days,Total value invoices paid between 31 and 60 days,Total value invoices paid later than 60 days,% Invoices paid within 30 days,% Invoices paid between 31 and 60 days,% Invoices paid later than 60 days,Total value invoices paid later than agreed terms,% Invoices not paid within agreed terms,% Invoices not paid due to dispute,Shortest (or only) standard payment period,Longest standard payment period,Maximum contractual payment period,Retention clauses included in all construction contracts,Retention clauses included in standard payment terms,Retention clauses are used in specific circumstances,Retention clauses used above a specific sum,Value above which retention clauses are used,Retention clauses are at a standard rate,Retention clauses standard rate percentage,Retention clauses have parity with client,Retention clause money release is staged,Retention value compared to client retentions as %,Retention value compared to total payments as %,Participates in payment codes,E-Invoicing offered,Supply-chain financing offered,Policy covers charges for remaining on supplier list,Charges have been made for remaining on supplier list,URL
0,2,Regime-1,NaT,2017-04-29,2017-10-28,2017-11-07,MEDTRONIC LIMITED,01070807,True,False,False,False,25.00,NaN,NaN,NaN,77.00,20.00,3.00,0.00,89.00,0.00,30.00,60.00,60.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,True,False,False,False,https://check-payment-practices.service.gov.uk/report/2
1,3,Regime-1,NaT,2017-05-01,2017-10-31,2017-11-08,SEBDEN STEEL SERVICE CENTRES LIMITED,02553464,True,False,False,False,69.00,NaN,NaN,NaN,7.00,28.00,65.00,0.00,0.00,0.00,30.00,NaN,75.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,False,False,https://check-payment-practices.service.gov.uk/report/3
2,4,Regime-1,NaT,2017-04-15,2017-10-14,2017-11-08,GREENERGY FLEXIGRID LIMITED,07581532,True,False,False,False,21.00,NaN,NaN,NaN,89.00,9.00,2.00,0.00,47.00,0.00,2.00,30.00,30.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,True,True,https://check-payment-practices.service.gov.uk/report/4
3,5,Regime-1,NaT,2017-04-15,2017-10-14,2017-11-08,GREENERGY FUELS LIMITED,04058825,True,False,False,False,14.00,NaN,NaN,NaN,90.00,8.00,2.00,0.00,31.00,0.00,2.00,30.00,30.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,True,True,https://check-payment-practices.service.gov.uk/report/5
4,6,Regime-1,NaT,2017-04-15,2017-10-14,2017-11-08,GREENERGY INTERNATIONAL LIMITED,02809935,True,False,False,False,15.00,NaN,NaN,NaN,96.00,2.00,2.00,0.00,14.00,0.00,2.00,30.00,30.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,True,False,https://check-payment-practices.service.gov.uk/report/6


In [6]:
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
pd.DataFrame({"dtype": df.dtypes.astype(str), "blanks": df.isna().sum()})

Shape: 115,358 rows, 42 columns


,dtype,blanks
Report Id,int64,0
Policy Regime,str,0
Financial period start date,datetime64[us],95331
Start date,datetime64[us],0
End date,datetime64[us],0
Filing date,datetime64[us],0
Company,str,0
Company number,str,0
Qualifying contracts in reporting period,bool,0
Payments made in reporting period,bool,0


In [7]:
# I find each column the analysis uses by a phrase that matches its name and
# no other, so no name with a % sign or a space is typed in full by hand.
PHRASES = {
    "regime": "Regime",
    "company": "Company number",
    "start": "Start date",
    "end": "End date",
    "days": "Average time",
    "within_30": "Invoices paid within",
    "31_to_60": "Invoices paid between",
    "over_60": "Invoices paid later",
    "outcome": "not paid within agreed",
    "dispute": "due to dispute",
    "qualifying": "Qualifying contracts",
    "codes": "payment codes",
    "filing": "Filing date",
    "id": "Report Id",
}
col = {}
for role, phrase in PHRASES.items():
    found = [name for name in df.columns if phrase in name]
    if len(found) != 1:
        raise KeyError(f"{phrase!r} matches {found}, not one column")
    col[role] = found[0]
pd.Series(col, name="column in the data").to_frame()

,column in the data
regime,Policy Regime
company,Company number
start,Start date
end,End date
days,Average time to pay
within_30,% Invoices paid within 30 days
31_to_60,% Invoices paid between 31 and 60 days
over_60,% Invoices paid later than 60 days
outcome,% Invoices not paid within agreed terms
dispute,% Invoices not paid due to dispute


**What loaded.** The file loads in one call as 115,358 reports and 52 columns. I drop the 10 free-text columns straight away, before anything is shown, which leaves every row and 42 columns. The four date columns parse as dates, the payment figures load as numbers, and `Company number` stays as text, which keeps its leading zeros. The 11 flag columns with blanks loaded as object, so I gave them pandas' nullable boolean type; the other four have no blanks and loaded as bool. `Report Id` is an identifier, not a measure.

**Blanks.** The payment figures are blank in 9,500 reports, and the share of invoices paid late in 9,423. `Financial period start date` is blank in 95,331, and the retention clause values and percentages are almost wholly blank; I use none of these. The anomaly list at the end of this section looks at the blanks the analysis depends on.

### Task 3: Descriptive statistics

I describe all 42 columns as loaded, before any anomaly is ruled on, so these figures still include the anomalies listed at the end of this section. There are no charts here; the only charts are the three visual models and the fitted model.

In [8]:
df.describe(include="number").T

,count,mean,std,min,25%,50%,75%,max
Report Id,"115,358.00","58,786.74","33,799.42",2.00,"29,575.25","58,848.50","88,060.75","117,289.00"
Average time to pay,"105,858.00",112.91,"17,717.22",0.00,24.00,33.00,45.00,"4,323,394.00"
Total value invoices paid within 30 days,"14,380.00","121,967,136.34","721,055,611.76",0.00,"5,653,401.25","17,134,562.00","54,210,525.00","29,157,881,704.00"
Total value invoices paid between 31 and 60 days,"14,380.00","70,306,816.78","2,092,110,316.07",0.00,"1,380,845.00","6,450,343.00","22,067,426.75","240,120,146,721.00"
Total value invoices paid later than 60 days,"14,380.00","22,671,812.06","184,971,990.51",0.00,"222,367.25","1,604,357.50","8,343,486.25","15,886,037,312.00"
% Invoices paid within 30 days,"105,858.00",56.97,28.69,0.00,32.00,60.00,83.00,100.00
% Invoices paid between 31 and 60 days,"105,858.00",29.53,20.79,0.00,12.00,27.00,44.00,100.00
% Invoices paid later than 60 days,"105,858.00",13.32,16.65,0.00,2.00,7.00,18.00,100.00
Total value invoices paid later than agreed terms,"101,621.00","3,528,043.38","38,371,080.63",0.00,0.00,0.00,0.00,"4,279,246,244.00"
% Invoices not paid within agreed terms,"105,935.00",26.22,23.27,0.00,8.00,20.00,39.00,100.00


**What it shows.** The share of invoices paid late has a mean of 26.22% and a median of 20%, and a quarter of reports are at 39% or more, so it leans to the right. Average time to pay has a median of 33 days but a mean of 112.91 and a maximum of 4,323,394 days, more than eleven thousand years. A few values that look like typing errors pull the mean far above the typical report, so I describe it by its median. The three time bands have medians of 60%, 27% and 7%. The payment period columns reach 1,000, 1,460 and 5,475 days, which also look like typing errors, and the money columns are filled in only 14,380 reports; I use neither. `Report Id` is an identifier, so its figures mean nothing.

In [9]:
df[col["regime"]].value_counts()

Policy Regime
Regime-1    99861
Regime-3     8063
Regime-2     7434
Name: count, dtype: int64

**Policy Regime.** Regime-1 holds 99,861 of the 115,358 reports. The publisher does not define the regimes. From the dates and the regulations I infer that Regime-1 reports are the ones filed under the original 2017 rules, and that Regime-2 (7,434 reports) and Regime-3 (8,063) are filed under later amendments to those rules. The question uses Regime-1 reports only, so every figure it compares is measured under one definition.

In [10]:
flags = df.select_dtypes(include=["bool", "boolean"]).columns
flag_table = pd.DataFrame(
    {
        "True": df[flags].eq(True).sum(),
        "False": df[flags].eq(False).sum(),
        "blank": df[flags].isna().sum(),
    }
)
flag_table

,True,False,blank
Qualifying contracts in reporting period,105935,9423,0
Payments made in reporting period,97715,17643,0
Qualifying construction contracts in reporting period,1979,113379,0
Construction contracts have retention clauses,1138,114220,0
Retention clauses included in all construction contracts,297,105638,9423
Retention clauses included in standard payment terms,150,105785,9423
Retention clauses are used in specific circumstances,752,105183,9423
Retention clauses used above a specific sum,94,105841,9423
Retention clauses are at a standard rate,608,105327,9423
Retention clauses have parity with client,450,105485,9423


**Flag columns.** 9,423 reports say the company had no qualifying contracts in the period. That is the same number as the blanks in the share of invoices paid late and in the other flag columns, which suggests they are the same reports; the anomaly list checks it. E-invoicing is offered in 27,654 reports and supply-chain financing in 7,592. `Payments made in reporting period` is False in 17,643 reports, more than the 9,423 without qualifying contracts, so it does not mark the same reports, and I do not use it. Only 1,979 reports had qualifying construction contracts, which is why the retention clause flags are almost all False.

In [11]:
# The column is never blank: a company in no payment code answers "No". So I
# group it as "No" against any other answer, and print no raw answer.
answers_no = df[col["codes"]].str.strip().str.casefold().eq("no")
pd.Series(
    {"No": answers_no.sum(), "any other answer": (~answers_no).sum()},
    name="reports",
)

No                  104362
any other answer     10996
Name: reports, dtype: int64

**Payment codes.** 104,362 reports answer No and 10,996 give any other answer, about one report in ten. The answers are typed by hand, so I read the column only as No against any other answer. It is not part of the question; I describe it because it is one of the dataset's grouping variables.

In [12]:
# The five figures the question and the fitted model rest on. Their plausible
# limit is 100 for a percentage and an assumed 120 days for the average time to
# pay, well above its 99th percentile.
key = [
    col["days"],
    col["within_30"],
    col["31_to_60"],
    col["over_60"],
    col["outcome"],
]
limits = pd.Series(100, index=key)
limits[col["days"]] = 120
quantiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
distributions = df[key].quantile(quantiles).T
distributions.columns = [f"{q:.0%}" for q in quantiles]
distributions["skew"] = df[key].skew()
distributions["limit"] = limits
distributions["above limit"] = df[key].gt(limits).sum()
distributions

,1%,5%,25%,50%,75%,95%,99%,skew,limit,above limit
Average time to pay,4.00,11.00,24.00,33.00,45.00,67.15,91.00,231.41,120,358
% Invoices paid within 30 days,3.00,10.00,32.00,60.00,83.00,98.00,100.00,-0.21,100,0
% Invoices paid between 31 and 60 days,0.00,1.00,12.00,27.00,44.00,67.00,81.00,0.53,100,0
% Invoices paid later than 60 days,0.00,0.00,2.00,7.00,18.00,51.00,74.00,1.97,100,0
% Invoices not paid within agreed terms,0.00,0.00,8.00,20.00,39.00,76.00,93.00,1.06,100,0


**Distributions.** Average time to pay has a skew of 231.41 because of the few huge values. Between the 1st and 99th percentiles it runs from 4 to 91 days, and 358 reports are above the assumed 120-day limit, which I list as anomalies below. None of the four percentages goes above 100. The share of invoices paid late runs from 0% at the 5th percentile to 76% at the 95th, with a median of 20% and a skew of 1.06, so in most reports most invoices are paid on time, but a long tail of reports shows many paid late. The share paid within 30 days is the other way round, with a median of 60% and a slight left skew (-0.21).

### The anomaly list

Before any analysis uses the data, I list every anomaly I test for with one function, `find_anomalies`. Each row of the anomaly list gives the anomaly, how many reports it touches, its effect on the analysis and the rule that found it. Nothing is removed here. I decide on each anomaly (flag, keep or exclude) before the analysis table is built, and the analysis keeps every raw row.

In [13]:
def find_anomalies(
    frame,
    columns,
    percent_columns,
    periods,
    years=(2017, 2026),
    day_limits=(365, 120),
    band_tolerance=2,
    regime="Regime-1",
):
    """Return the anomaly list for frame, one row per check, in a fixed order.

    Each row gives the anomaly, how many rows it touches, its effect on the
    analysis and the rule that found it. A check that finds nothing keeps
    its row, with 0 rows and "None found.", so the list shows every check
    that ran. frame is not changed and no row is removed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per report.
    columns : dict of str to str
        Role to column name, for the roles company, start, end, regime,
        days, within_30, 31_to_60, over_60, outcome, dispute and qualifying.
    percent_columns : list of str
        The columns that hold a percentage.
    periods : dict of str to (int, int)
        Period name to its first and last year. A report falls in the
        period its end date falls in.
    years : (int, int), default (2017, 2026)
        The first and last year a start or end date may fall in.
    day_limits : tuple of int, default (365, 120)
        Each limit gets its own row: average time to pay above it.
    band_tolerance : float, default 2
        How far, in percentage points, the three time bands may sum from
        100 before the report is listed.
    regime : str, default "Regime-1"
        The regime whose reports should not carry a dispute percentage.

    Returns
    -------
    pandas.DataFrame
        Columns anomaly, rows, effect and rule, indexed from 1.
    """
    c = columns
    days = frame[c["days"]]
    start, end = frame[c["start"]], frame[c["end"]]
    found = []

    for limit in day_limits:
        over = days > limit
        found.append(
            (
                f"Average time to pay above {limit} days",
                over,
                f"The largest is {days[over].max():,.0f} days. The mean time "
                f"to pay is {days.mean():,.1f} days with these rows and "
                f"{days[~over].mean():,.1f} without them.",
                f"{c['days']} > {limit}",
            )
        )

    first, last = years
    start_out = (start.dt.year < first) | (start.dt.year > last)
    end_out = (end.dt.year < first) | (end.dt.year > last)
    found.append(
        (
            f"Start or end date outside {first} to {last}",
            start_out | end_out,
            f"{start_out.sum():,} start dates and {end_out.sum():,} end "
            f"dates; the earliest start is {start.min():%Y-%m-%d} and the "
            f"latest end {end.max():%Y-%m-%d}. A wrong end date can put a "
            "report in the wrong period.",
            f"year of {c['start']} or {c['end']} before {first} or after "
            f"{last}",
        )
    )
    found.append(
        (
            "End date before start date",
            end < start,
            "A report cannot end before it starts.",
            f"{c['end']} < {c['start']}",
        )
    )

    shares = frame[percent_columns]
    found.append(
        (
            "Payment percentage outside 0 to 100",
            ((shares < 0) | (shares > 100)).any(axis=1),
            "A share of invoices must lie between 0 and 100.",
            f"any of the {len(percent_columns)} percentage columns < 0 or "
            "> 100",
        )
    )

    band_names = [c["within_30"], c["31_to_60"], c["over_60"]]
    bands = frame[band_names]
    # min_count=3 leaves the sum blank unless all three bands are filled.
    total = bands.sum(axis=1, min_count=3)
    off = (total - 100).abs() > band_tolerance
    low, high = total[off].min(), total[off].max()
    spread = (
        f"They all sum to {low:,.0f}"
        if low == high
        else f"Their sums run from {low:,.0f} to {high:,.0f}"
    )
    found.append(
        (
            "Three time bands do not sum to about 100",
            off,
            f"{spread}, and {(bands[off] == 0).all(axis=1).sum():,} have 0 "
            "in all three bands.",
            f"|{' + '.join(band_names)} - 100| > {band_tolerance}",
        )
    )

    key = [c["company"], c["start"], c["end"]]
    figures = [c["days"], *band_names, c["outcome"]]
    repeated = frame.duplicated(key, keep=False)
    found.append(
        (
            "Duplicate report: same company, same reporting period",
            repeated,
            f"{len(frame.loc[repeated, key].drop_duplicates()):,} company "
            "periods are reported more than once "
            f"({frame.duplicated(key).sum():,} rows beyond the first), and "
            f"{frame.duplicated(key + figures).sum():,} of those repeat the "
            "figures exactly. Each repeat would count that period again in "
            "the company's mean.",
            f"same {', '.join(key)}",
        )
    )

    several = pd.Series(False, index=frame.index)
    pairs, companies = 0, set()
    for first_year, last_year in periods.values():
        inside = end.dt.year.between(first_year, last_year)
        sizes = frame.loc[inside, c["company"]].value_counts()
        many = sizes[sizes > 1].index
        several |= inside & frame[c["company"]].isin(many)
        pairs += len(many)
        companies |= set(many)
    found.append(
        (
            "Company with more than one report in a period",
            several,
            f"{pairs:,} company periods across {len(companies):,} companies. "
            "Unless each company's reports are averaged to one figure per "
            "period, a company that reports more often counts more.",
            f"same {c['company']} on more than one report in a period, "
            f"where a report's period is the one its {c['end']} falls in ("
            + " or ".join(f"{a} to {b}" for a, b in periods.values())
            + ")",
        )
    )

    blank = frame[c["outcome"]].isna()
    no_contracts = (blank & frame[c["qualifying"]].eq(False)).sum()
    found.append(
        (
            "Share of invoices paid late is blank",
            blank,
            f"{no_contracts:,} of them say the company had no qualifying "
            "contracts in the period. These reports cannot enter the "
            "analysis.",
            f"{c['outcome']} is blank",
        )
    )

    disputed = frame[c["regime"]].eq(regime) & frame[c["dispute"]].notna()
    found.append(
        (
            f"{regime} report with a dispute percentage",
            disputed,
            f"{(disputed & frame[c['dispute']].eq(0)).sum():,} of them are "
            "exactly 0. The rules asked for this figure only from "
            "SI 2024/444, so the analysis does not use it.",
            f"{c['regime']} is {regime} and {c['dispute']} is not blank",
        )
    )

    table = pd.DataFrame(
        [
            {
                "anomaly": label,
                "rows": int(mask.sum()),
                "effect": effect if mask.any() else "None found.",
                "rule": rule,
            }
            for label, mask, effect, rule in found
        ]
    )
    table.index = pd.RangeIndex(1, len(table) + 1, name="#")
    return table

In [14]:
# The limits are the ones fixed for this project: dates within 2017 to 2026,
# an assumed 120 days with 365 as the extreme case, and 2 points either side
# of 100 for the bands, which are published as whole numbers.
anomalies = find_anomalies(
    df,
    col,
    percent_columns=[name for name in df.columns if name.startswith("%")],
    periods={"2018 to 2019": (2018, 2019), "2023 to 2024": (2023, 2024)},
    years=(2017, 2026),
    day_limits=(365, 120),
    band_tolerance=2,
)
anomalies

,anomaly,rows,effect,rule
#,,,,
1,Average time to pay above 365 days,49,"The largest is 4,323,394 days. The mean time to pay is 112.9 days with these rows and 35.8 without them.",Average time to pay > 365
2,Average time to pay above 120 days,358,"The largest is 4,323,394 days. The mean time to pay is 112.9 days with these rows and 35.4 without them.",Average time to pay > 120
3,Start or end date outside 2017 to 2026,34,29 start dates and 5 end dates; the earliest start is 1918-12-30 and the latest end 3025-06-30. A wrong end date can put a report in the wrong period.,year of Start date or End date before 2017 or after 2026
4,End date before start date,0,None found.,End date < Start date
5,Payment percentage outside 0 to 100,0,None found.,any of the 5 percentage columns < 0 or > 100
6,Three time bands do not sum to about 100,178,"They all sum to 0, and 178 have 0 in all three bands.",|% Invoices paid within 30 days + % Invoices paid between 31 and 60 days + % Invoices paid later than 60 days - 100| > 2
7,"Duplicate report: same company, same reporting period",3782,"1,861 company periods are reported more than once (1,921 rows beyond the first), and 223 of those repeat the figures exactly. Each repeat would count that period again in the company's mean.","same Company number, Start date, End date"
8,Company with more than one report in a period,53860,"14,417 company periods across 9,009 companies. Unless each company's reports are averaged to one figure per period, a company that reports more often counts more.","same Company number on more than one report in a period, where a report's period is the one its End date falls in (2018 to 2019 or 2023 to 2024)"
9,Share of invoices paid late is blank,9423,"9,423 of them say the company had no qualifying contracts in the period. These reports cannot enter the analysis.",% Invoices not paid within agreed terms is blank


**What the anomaly list shows.** Two checks find nothing: no report ends before it starts, and no percentage is outside 0 to 100. The rest need a decision before the analysis table is built.

- **Days.** The 49 reports above 365 days, the largest at 4,323,394 days, lift the mean time to pay from 35.8 to 112.9 days on their own. They matter for the fitted model, which uses average time to pay; 358 reports are above the assumed 120-day limit.
- **Dates.** 29 start dates before 2017 (as early as 1918) and 5 end dates after 2026 (as late as 3025) look like typing errors, and a wrong end date can put a report in the wrong period.
- **Bands.** The 178 reports whose three time bands do not sum to about 100 all sum to 0, with 0 in every band.
- **Repeats.** 1,861 company periods are reported more than once, and only 223 of the 1,921 extra rows repeat the figures exactly, so I need a rule for which copy counts.
- **More than one report in a period.** This is normal, because a company reports at least twice a year; one figure per company per period deals with it.
- **Blanks.** All 9,423 blank values of the share of invoices paid late come from reports that say the company had no qualifying contracts, so they are empty reports, not missing answers.
- **Disputes.** 87,186 of the 87,188 Regime-1 dispute percentages are exactly 0, which looks like a default value, not a reported figure. The analysis does not use the field.

### The analysis table

Here I apply my ruling on each anomaly in the list above, plus one more check, for figures that cannot all be true. Each ruling marks the reports it covers with a `flag` and a `flag_reason`, and no raw row is deleted. Some rulings exclude reports from the analysis, one excludes them from the fitted model only, and the rest keep them.

Then I build the analysis table: Regime-1 reports only, each placed in a period by the year its reporting period ends, one unweighted mean of the share of invoices paid late per company per period, and only the companies with a figure in both periods. I show every count with and without the excluded rows.

In [15]:
def find_impossible_figures(frame, columns, rounding=0.5, tail_days=3650):
    """Return two checks for reports whose own figures cannot all be true.

    The average time to pay and the three time bands describe the same
    payments, so the bands bound the average. Each figure is published as a
    whole number, so I allow `rounding` either way before calling a report
    impossible.

    - below the floor: the average is lower than the bands allow, with the
      first band starting at day 0, the second at day 31 and the third at
      day 61.
    - above the ceiling: no payment is reported after day 60, yet the
      average is so high that the hidden share of under `rounding` per cent
      would need to take more than `tail_days` days each.

    Reports with all three bands at 0 are left out of both checks, as
    ruled, and a report with a blank figure is never flagged. frame is not
    changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per report.
    columns : dict of str to str
        Role to column name, for days, within_30, 31_to_60 and over_60.
    rounding : float, default 0.5
        How far a published figure may sit from its true value.
    tail_days : float, default 3650
        The longest average, in days, allowed for the hidden share.

    Returns
    -------
    pandas.DataFrame
        Boolean columns "below the floor" and "above the ceiling", on
        frame's index.
    """
    c = columns
    days = frame[c["days"]]
    second, third = frame[c["31_to_60"]], frame[c["over_60"]]
    bands = frame[[c["within_30"], c["31_to_60"], c["over_60"]]]
    all_zero = bands.eq(0).all(axis=1)
    # The fewest days the bands allow: each share rounded down and paid on
    # the first day of its band.
    floor = (
        31 * (second - rounding).clip(lower=0)
        + 61 * (third - rounding).clip(lower=0)
    ) / 100
    below = (days + rounding) < floor
    # The most the first two bands can carry: each share rounded up and paid
    # on the last day of its band. The rest is the hidden share.
    hidden = rounding / 100
    middle = ((second + rounding) / 100).clip(upper=1 - hidden)
    first = 1 - hidden - middle
    needed = ((days - rounding) - 30 * first - 60 * middle) / hidden
    above = third.eq(0) & (needed > tail_days)
    return pd.DataFrame(
        {
            "below the floor": below & ~all_zero,
            "above the ceiling": above & ~all_zero,
        }
    )

In [16]:
impossible = find_impossible_figures(df, col)
either = impossible.any(axis=1)
print(f"Below the floor: {impossible['below the floor'].sum():,} reports")
print(f"Above the ceiling: {impossible['above the ceiling'].sum():,} reports")
print(
    f"Either check: {either.sum():,} reports from "
    f"{df.loc[either, col['company']].nunique():,} companies"
)

Below the floor: 355 reports
Above the ceiling: 15 reports
Either check: 370 reports from 236 companies


**Figures that cannot all be true.** The average time to pay and the three time bands describe the same payments, so the bands set limits on the average. 355 reports give an average lower than their own bands allow, and 15 report nothing paid after day 60 yet give an average that the hidden sliver of payments could reach only by taking more than ten years each. Together that is 370 reports from 236 companies. I allow half a unit of rounding on every figure, so each of these is impossible as published, not merely unlikely. I leave out the reports with all three bands at 0, as ruled.

In [17]:
def apply_rulings(
    frame,
    columns,
    impossible,
    years=(2017, 2026),
    day_limits=(365, 120),
    regime="Regime-1",
):
    """Return a copy of frame with my ruling on each anomaly as a flag.

    Two columns are added: flag, True when any ruling covers the report,
    and flag_reason, the rulings that cover it in ruling order, joined by
    "; " and blank when none does. No row is removed, so every total can be
    shown with and without the flagged reports. frame is not changed.

    The rulings, in order:

    1. Average time to pay above the upper day limit.
    2. Average time to pay above the lower day limit, up to the upper one.
    3. Start or end date outside the years.
    4. All three time bands are 0.
    5. Superseded by a later report for the same period: where a company
       has more than one report with the same start and end date, the
       latest-filed report stands, and a same-day tie goes to the higher
       Report Id, which the service gives out in filing order.
    6. Share of invoices paid late is blank.
    7. A report of the regime with a dispute percentage.
    8. Figures that cannot all be true, as marked in impossible.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per report.
    columns : dict of str to str
        Role to column name, for id, company, start, end, filing, regime,
        days, within_30, 31_to_60, over_60, outcome and dispute.
    impossible : pandas.Series of bool
        On frame's index: the reports whose figures cannot all be true.
    years : (int, int), default (2017, 2026)
        The first and last year a start or end date may fall in.
    day_limits : (int, int), default (365, 120)
        The upper and lower limits for average time to pay, in days.
    regime : str, default "Regime-1"
        The regime whose reports should not carry a dispute percentage.

    Returns
    -------
    pandas.DataFrame
        frame's columns plus flag and flag_reason.
    """
    c = columns
    days = frame[c["days"]]
    upper, lower = day_limits
    first, last = years
    start_year, end_year = frame[c["start"]].dt.year, frame[c["end"]].dt.year
    bands = frame[[c["within_30"], c["31_to_60"], c["over_60"]]]
    in_filing_order = frame.sort_values([c["filing"], c["id"]])
    superseded = in_filing_order.duplicated(
        [c["company"], c["start"], c["end"]], keep="last"
    ).reindex(frame.index)
    rulings = [
        (f"Average time to pay above {upper} days", days > upper),
        (
            f"Average time to pay above {lower} days, up to {upper}",
            (days > lower) & (days <= upper),
        ),
        (
            f"Start or end date outside {first} to {last}",
            (start_year < first)
            | (start_year > last)
            | (end_year < first)
            | (end_year > last),
        ),
        ("All three time bands are 0", bands.eq(0).all(axis=1)),
        ("Superseded by a later report for the same period", superseded),
        ("Share of invoices paid late is blank", frame[c["outcome"]].isna()),
        (
            f"{regime} report with a dispute percentage",
            frame[c["regime"]].eq(regime) & frame[c["dispute"]].notna(),
        ),
        ("Figures that cannot all be true", impossible),
    ]
    reasons = pd.Series("", index=frame.index)
    for reason, covered in rulings:
        reasons = reasons.mask(covered, reasons + reason + "; ")
    out = frame.copy()
    out["flag_reason"] = reasons.str.removesuffix("; ")
    out["flag"] = out["flag_reason"].ne("")
    return out[[*frame.columns, "flag", "flag_reason"]]


def has_reason(frame, reasons):
    """Return True for each report whose flag_reason includes any of reasons.

    A reason matches only whole, so "A" does not match "AB". frame is not
    changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        A frame from apply_rulings, with its flag_reason column.
    reasons : list of str
        The rulings to look for.

    Returns
    -------
    pandas.Series of bool
        On frame's index.
    """
    parts = frame["flag_reason"].str.split("; ").explode()
    found = parts.isin(reasons).groupby(level=0).any()
    return found.reindex(frame.index)

In [18]:
flagged = apply_rulings(df, col, either)
# What each ruling does with the reports it flags. The first takes reports
# out of the fitted model only, whose input is the average time to pay.
RULINGS = {
    "Average time to pay above 365 days": "Excluded from the fitted model",
    "Average time to pay above 120 days, up to 365": "Kept",
    "Start or end date outside 2017 to 2026": "Excluded",
    "All three time bands are 0": "Kept, left out of the band-based checks",
    "Superseded by a later report for the same period": (
        "Excluded, the latest-filed report stands"
    ),
    "Share of invoices paid late is blank": "Excluded, no outcome to measure",
    "Regime-1 report with a dispute percentage": (
        "Kept, the dispute percentage is not used"
    ),
    "Figures that cannot all be true": (
        "Excluded, results shown with and without"
    ),
}
EXCLUDED = [
    reason
    for reason, use in RULINGS.items()
    if use == "Excluded" or use.startswith("Excluded,")
]
rulings = pd.DataFrame(
    {
        "rows": [has_reason(flagged, [reason]).sum() for reason in RULINGS],
        "use": list(RULINGS.values()),
    },
    index=pd.Index(list(RULINGS), name="ruling"),
)
rulings

,rows,use
ruling,,
Average time to pay above 365 days,49,Excluded from the fitted model
"Average time to pay above 120 days, up to 365",309,Kept
Start or end date outside 2017 to 2026,34,Excluded
All three time bands are 0,178,"Kept, left out of the band-based checks"
Superseded by a later report for the same period,1921,"Excluded, the latest-filed report stands"
Share of invoices paid late is blank,9423,"Excluded, no outcome to measure"
Regime-1 report with a dispute percentage,87188,"Kept, the dispute percentage is not used"
Figures that cannot all be true,370,"Excluded, results shown with and without"


**The rulings.** Each row is one ruling, with the number of reports it flags and what it does with them. Four rulings exclude reports from the analysis: the 34 with a start or end date outside 2017 to 2026, which could put a report in the wrong period; the 1,921 superseded reports; the 9,423 with no share of invoices paid late; and the 370 whose figures cannot all be true. Where a company filed more than one report for the same period, I keep the latest-filed one, as the government's own statistics do, so the 3,782 duplicate rows lose 1,921 and keep one for each of the 1,861 company periods. The 49 reports above 365 days stay in the test but leave the fitted model, whose input is the average time to pay; with the 309 between 120 and 365 days, they make the 358 that the anomaly list found above 120. I keep the 178 reports with all three bands at 0, and the analysis never uses the 87,188 Regime-1 dispute percentages.

In [19]:
def assign_period(frame, columns, periods):
    """Return the period each report falls in, by the year its period ends.

    A report whose end date falls in none of the periods gets a blank.
    frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per report.
    columns : dict of str to str
        Role to column name, for end.
    periods : dict of str to (int, int)
        Period name to its first and last year.

    Returns
    -------
    pandas.Series
        The period name, or a blank, on frame's index.
    """
    end_year = frame[columns["end"]].dt.year
    period = pd.Series(pd.NA, index=frame.index, dtype="str")
    for name, (first, last) in periods.items():
        period = period.mask(end_year.between(first, last), name)
    return period


def build_analysis_table(frame, columns, periods, regime="Regime-1"):
    """Return the analysis table and the count left after each step.

    The table has one row per company with a figure in every period: the
    unweighted mean of its reports' share of invoices paid late in each
    period, then the change from the first period to the last. A report is
    placed in a period by assign_period. Pass frame with any excluded
    reports already removed. frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per report.
    columns : dict of str to str
        Role to column name, for company, end, regime and outcome.
    periods : dict of str to (int, int)
        Period name to its first and last year, earliest first.
    regime : str, default "Regime-1"
        The only regime whose reports are used.

    Returns
    -------
    table : pandas.DataFrame
        Indexed by company number, one column per period, then change.
    steps : pandas.Series
        The reports, company periods or companies left after each step.
    """
    c = columns
    names = list(periods)
    in_regime = frame[frame[c["regime"]].eq(regime)]
    period = assign_period(in_regime, c, periods)
    in_period = in_regime[period.notna()]
    measured = in_period[in_period[c["outcome"]].notna()]
    means = (
        measured.assign(period=period)
        .groupby([c["company"], "period"])[c["outcome"]]
        .mean()
    )
    wide = means.unstack().reindex(columns=names)
    table = wide.dropna().rename_axis(columns=None)
    table["change"] = table[names[-1]] - table[names[0]]
    steps = {
        "Reports": len(frame),
        f"{regime} reports": len(in_regime),
        f"{regime} reports ending in " + " or ".join(names): len(in_period),
        "Of those, with a share of invoices paid late": len(measured),
        "Company periods, one mean each": len(means),
    }
    for name in names:
        steps[f"Companies in {name}"] = int(wide[name].notna().sum())
    steps["Paired companies, in both periods"] = len(table)
    return table, pd.Series(steps)

In [20]:
# A report is placed in a period by the year its reporting period ends.
PERIODS = {"2018 to 2019": (2018, 2019), "2023 to 2024": (2023, 2024)}
excluded = has_reason(flagged, EXCLUDED)
every_report, every_steps = build_analysis_table(flagged, col, PERIODS)
paired, kept_steps = build_analysis_table(flagged[~excluded], col, PERIODS)
if len(paired) < 200:
    raise ValueError(
        f"Only {len(paired)} paired companies, under the 200 the analysis "
        "needs. Stop and report this before going on."
    )
steps = pd.DataFrame(
    {"every report": every_steps, "without the excluded rows": kept_steps}
)
print(f"Excluded rows: {excluded.sum():,} reports")
print(f"Paired companies: {len(paired):,} (the analysis needs at least 200)")
steps

Excluded rows: 10,271 reports
Paired companies: 5,371 (the analysis needs at least 200)


,every report,without the excluded rows
Reports,115358,105087
Regime-1 reports,99861,90792
Regime-1 reports ending in 2018 to 2019 or 2023 to 2024,54759,49512
"Of those, with a share of invoices paid late",49895,49512
"Company periods, one mean each",14255,14242
Companies in 2018 to 2019,7476,7468
Companies in 2023 to 2024,6779,6774
"Paired companies, in both periods",5376,5371


**The analysis table.** With every report, 54,759 Regime-1 reports end in one of the two periods, 49,895 of them with a share of invoices paid late, and 5,376 companies have a figure in both periods. The four excluding rulings take out 10,271 reports in all, which leaves 49,512 reports and 5,371 paired companies, well above the 200 the analysis needs. So the exclusions cost only five companies: a superseded report always leaves its latest-filed twin behind, and few companies have only impossible or misdated reports in a period. Each company counts once per period, however many reports it filed, so a company that reports more often does not weigh more. I use the 5,371 paired companies from here on.

## Model definition

## Training

## Evaluation

## Summary